# Using GPU for PyTorch

If you are on a local device or a cluster, and the GPU option is available, you can usually use the GPU version of PyTorch.

On Google Colab, the default backend is CPU. You need to manually request GPU resource:

Click on `Runtime` on the top right menu, choose `Change runtime type`. Then you can choose GPU in the dialog window.

After this, your Notebook will restart and you are on the GPU node.

Google Colab uses Nvidia GPU supported by CUDA.

In [1]:
import torch
import torch.nn as nn # the nn module in torch contains functions and classes for neural networks
from torch import optim # optim provides optimization modules to optimize the NN parameters

In [2]:
# See if GPU is available
print("CUDA available:", torch.cuda.is_available()) # you should see True if you choose the GPU backend

CUDA available: True


In [3]:
# See details of the hardware - only when you are on GPU
!nvidia-smi

Tue Feb 10 23:57:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### If you want to use GPU for your task, you need to specify `device`

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


##1.  Repeat the simplest case in Lab 7-1, but on GPU

Rule:

1. Move dataset (X and y) and the model to the specified device.
2. Other part remains the same

In [5]:
torch.manual_seed(0)  # for reproducibility

# Data (move to device)
x = torch.linspace(-3, 3, 2000).reshape(2000, 1).to(device)
y = torch.sin(x) + 0.1 * torch.randn_like(x) # since y is computed using x, y is already on the same device; otherwise, you need to add .to(device) explicitly

# Model (move to device)
model = nn.Sequential(
    nn.Linear(1, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
).to(device)


In [6]:
# Loss and optimizer
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
n_epoch = 1000
for epoch in range(n_epoch):
    optimizer.zero_grad()
    y_hat = model(x)              # forward pass
    loss = loss_fn(y_hat, y)
    loss.backward()               # backpropagation
    optimizer.step()              # update parameters

    if (epoch + 1) % 100 == 0 or epoch == 0:
        print(f"Epoch {epoch+1} | Loss = {loss.item():.6f}")

Epoch 1 | Loss = 0.921713
Epoch 100 | Loss = 0.190921
Epoch 200 | Loss = 0.135579
Epoch 300 | Loss = 0.090948
Epoch 400 | Loss = 0.056449
Epoch 500 | Loss = 0.034174
Epoch 600 | Loss = 0.022940
Epoch 700 | Loss = 0.017951
Epoch 800 | Loss = 0.015465
Epoch 900 | Loss = 0.013869
Epoch 1000 | Loss = 0.012787


In [7]:
# Prediction
model.eval()  # set model to evaluation mode
with torch.no_grad(): # turn off evaluating the gradient so it is faster
    x_test = torch.tensor([[-1.0], [0.0], [1.0]]).to(device)
    y_pred = model(x_test)
    print(y_pred)

tensor([[-0.8345],
        [-0.0034],
        [ 0.8834]], device='cuda:0')


## 2. The Class example

In [8]:
class SineApproximator(nn.Module): # This is standard: we inherit nn.Module class
    def __init__(self, input_dim=1, hidden_dim=32, output_dim=1, device="cpu"): # add a new argument device
        super().__init__() # super() is the parent class

        self.device = device

        # define the FNN model
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        ).to(self.device) # move model to device

    # forward passing
    def forward(self, x):
        return self.model(x)

    def train_model(self, x_train, y_train, n_epoch=1000, lr=1e-3):

        # move x_train and y_train to device
        x_train = x_train.to(self.device)
        y_train = y_train.to(self.device)

        ### The following is NOT affected by the choice of device
        # loss function and optimizer
        loss_fn = nn.MSELoss()
        optimizer = optim.Adam(self.parameters(), lr=lr)

        for epoch in range(n_epoch):
            optimizer.zero_grad()        # clear previous gradients
            y_hat = self(x_train)        # forward pass
            loss = loss_fn(y_hat, y_train)
            loss.backward()              # backpropagation
            optimizer.step()             # update parameters

            if (epoch+1) % 100 == 0 or epoch == 0:
                print(f"Epoch {epoch+1} | Loss = {loss.item():.6f}")

    def predict(self, x_test):
        self.eval()
        x_test = x_test.to(self.device)                   # evaluation mode
        with torch.no_grad():            # disable gradient computation
            return self(x_test)


In [9]:
# Create dataset
x = torch.linspace(-3, 3, 2000).reshape(-1, 1)      # (num_data, num_features)
y = torch.sin(x) + 0.1 * torch.randn_like(x)       # noisy sine wave

# Initialize model
my_model = SineApproximator(device=device) # specify device

# Train model
my_model.train_model(x, y, n_epoch=1000, lr=1e-3)

Epoch 1 | Loss = 0.369244
Epoch 100 | Loss = 0.127587
Epoch 200 | Loss = 0.060679
Epoch 300 | Loss = 0.030843
Epoch 400 | Loss = 0.020601
Epoch 500 | Loss = 0.016102
Epoch 600 | Loss = 0.013907
Epoch 700 | Loss = 0.012725
Epoch 800 | Loss = 0.012030
Epoch 900 | Loss = 0.011606
Epoch 1000 | Loss = 0.011301


In [11]:
my_model.predict(torch.tensor([[-1.0], [0.0], [1.0]]))

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)